In [20]:
!pip install -q langchain langchain-community langchain-huggingface chromadb langchain-chroma sentence-transformers

In [21]:
pip install langchain-core

In [22]:
import json
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load your JSON data
with open("/content/complete_windows.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 2. Build the list of Document objects
docs = []
for item in data:
    text = f"Content:{item['Content']}".strip()
    docs.append(Document(
        page_content=text,
        metadata={
            "source": item.get("source", ""),
            "id": item.get("id", "")
        }
    ))

# 3. Initialize HuggingFaceEmbeddings (using model_name)
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

# 4. Batch ingestion into Chroma
batch_size = 100
db = None

for i in range(0, len(docs), batch_size):
    print(f"Processing {i} -> {i + batch_size}")
    batch = docs[i:i + batch_size]

    if db is None:
        # Initialize the database with the first batch
        db = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            persist_directory="db_17_v17"
        )
    else:
        # Incrementally add subsequent batches to the existing database
        db.add_documents(documents=batch)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Processing 0 -> 100
Processing 100 -> 200
Processing 200 -> 300
Processing 300 -> 400
Processing 400 -> 500


In [25]:
!zip -r db_17_v17 /content/db_17_v17

  adding: content/db_17_v17/ (stored 0%)
  adding: content/db_17_v17/chroma.sqlite3 (deflated 58%)
  adding: content/db_17_v17/de9f3637-2069-4099-8ec3-d9888aff3a03/ (stored 0%)
  adding: content/db_17_v17/de9f3637-2069-4099-8ec3-d9888aff3a03/length.bin (deflated 57%)
  adding: content/db_17_v17/de9f3637-2069-4099-8ec3-d9888aff3a03/index_metadata.pickle (deflated 60%)
  adding: content/db_17_v17/de9f3637-2069-4099-8ec3-d9888aff3a03/link_lists.bin (deflated 84%)
  adding: content/db_17_v17/de9f3637-2069-4099-8ec3-d9888aff3a03/data_level0.bin (deflated 10%)
  adding: content/db_17_v17/de9f3637-2069-4099-8ec3-d9888aff3a03/header.bin (deflated 58%)


In [27]:
from google.colab import files
files.download("/content/db_17_v17.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>